# Setup

In [1]:
from langchain.prompts import PromptTemplate
from langchain import hub
from langchain.agents import Tool, AgentExecutor, initialize_agent, create_react_agent
from langchain.tools import DuckDuckGoSearchRun
from langchain_experimental.tools.python.tool import PythonREPLTool
from langchain_openai import ChatOpenAI
import yaml
import os

In [2]:
with open('config.yaml', 'r') as config_file:
    config = yaml.safe_load(config_file)
os.environ['OPENAI_API_KEY'] = config['OPENAI_API_KEY']

In [3]:
openai = ChatOpenAI(model_name='gpt-3.5-turbo', temperature=0)

# Assitente Financeiro Pessoal

In [4]:
prompt = '''
Como assistente financeiro pessoal, ajude a responder as seguintes perguntas com ajuda da internet.
Perguntas: {q}
'''
prompt_template = PromptTemplate.from_template(prompt)
print(prompt_template)

input_variables=['q'] template='\nComo assistente financeiro pessoal, ajude a responder as seguintes perguntas com ajuda da internet.\nPerguntas: {q}\n'


In [5]:
react_instructions  = hub.pull('hwchase17/react')
print(react_instructions )

input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'] metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'} template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}'


In [6]:
# Ferramenta 1 Python REPL
python_repl = PythonREPLTool()
python_repl_tool = Tool(
    name='Python REPL',
    func=python_repl.run,
    description='''Qualquer tipo de cálculo deve usar esta ferramenta. Você não deve realizar
                    o cálculo diretamente. Você deve inserir código Python.'''
)

In [7]:
# Ferramenta  2 busca DuckDuckGo 
search = DuckDuckGoSearchRun()
duckduckgo_tool = Tool(
    name='Busca DuckDuckGo',
    func=search.run,
    description='''Útil para encontrar informações e dicas de economia e opções de investimento.
                Você sempre deve pesquisar na internet as melhores dicas usando esta ferramenta, não
                responda diretamente. Sua resposta deve informar que há elementos pesquisados na internet'''
)

In [8]:
tools = [python_repl_tool, duckduckgo_tool]
agent = create_react_agent(openai, tools, react_instructions)

In [9]:
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=10
)
print(agent_executor)

verbose=True agent=RunnableAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_log_to_str(x['intermediate_steps']))
})
| PromptTemplate(input_variables=['agent_scratchpad', 'input'], partial_variables={'tools': "Python REPL(tool_input: 'Union[str, Dict[str, Any]]', verbose: 'Optional[bool]' = None, start_color: 'Optional[str]' = 'green', color: 'Optional[str]' = 'green', callbacks: 'Callbacks' = None, *, tags: 'Optional[List[str]]' = None, metadata: 'Optional[Dict[str, Any]]' = None, run_name: 'Optional[str]' = None, run_id: 'Optional[uuid.UUID]' = None, config: 'Optional[RunnableConfig]' = None, tool_call_id: 'Optional[str]' = None, **kwargs: 'Any') -> 'Any' - Qualquer tipo de cálculo deve usar esta ferramenta. Você não deve realizar\n                    o cálculo diretamente. Você deve inserir código Python.\nBusca DuckDuckGo(tool_input: 'Union[str, Dict[str, Any]]', verbose: 'Optional[bool]' = None, start_color: 'Optional[str]' = 'green', color

In [10]:
question = """
Minha renda é de R$6000 por mês, o total de minhas despesas é de R$2500 mais 1000 de aluguel.
Quais dicas de investimento você me dá?
"""

In [11]:
output = agent_executor.invoke({
    'input': prompt_template.format(q=question)
})



> Entering new AgentExecutor chain...
Eu devo buscar na internet dicas de investimento para ajudar a pessoa a tomar decisões financeiras mais informadas.
Action: Busca DuckDuckGo
Action Input: "melhores dicas de investimento para renda de R$6000 por mês"

C:\Users\jhone\anaconda3\envs\Langchain\Lib\site-packages\langchain_community\utilities\duckduckgo_search.py:64: UserWarning: backend='api' is deprecated, using backend='auto'
  ddgs_gen = ddgs.text(


Neste artigo, abordaremos dicas essenciais para identificar o melhor investimento para renda mensal, além de explorar as vantagens do investimento em imóveis, os fundos imobiliários, e a renda fixa como opções seguras. Conheça os investimentos que pagam mensalmente para turbinar seu orçamento e diversificar sua carteira de aplicações! Por isso, a seguir, você vai conhecer os melhores investimentos com renda mensal, entender como funcionam e quais são os que se adequam melhor ao seu perfil de investidor. Em outras palavras, não compensa aplicar todo o patrimônio apenas no investimento que parece render mais. É por essas e outras que iremos detalhar este assunto a seguir e te ajudar a escolher no que investir. Investir R$ 6.000,00 reais a 100,00% do CDI pode ser uma escolha interessante para quem busca segurança e rentabilidade moderada. Ao entender quanto rende essa aplicação e como calcular seus lucros, você pode fazer escolhas mais informadas e maximizar seus ganhos.As dicas de invest

In [12]:
print(output['input'])


Como assistente financeiro pessoal, ajude a responder as seguintes perguntas com ajuda da internet.
Perguntas: 
Minha renda é de R$6000 por mês, o total de minhas despesas é de R$2500 mais 1000 de aluguel.
Quais dicas de investimento você me dá?




In [13]:
print(output['output'])

As dicas de investimento incluem opções como investimento em imóveis, fundos imobiliários e renda fixa. Também é mencionado que investir R$ 6.000,00 a 100% do CDI pode ser uma escolha interessante para quem busca segurança e rentabilidade moderada.
